In [1]:
# 1. Import libraries
import torch
import torch.nn as nn
from torchvision import models, transforms

# 2. Define the number of classes
num_classes = 10

# 3. Load a pre-trained MobileNetV3-small model
print("Downloading pre-trained model...")
model = models.mobilenet_v3_small(pretrained=True)

# 4. Modify the classifier head for our specific number of classes
model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

# 5. Define basic image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 6. Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print("✅ Model initialized successfully!")
print(f"✅ Classifier head adapted for {num_classes} classes.")

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


✅ Model initialized successfully!
✅ Classifier head adapted for 10 classes.


In [2]:
from torchvision import datasets
from torch.utils.data import DataLoader

# Define transformations (resize to 224 for MobileNet)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3), # Convert 1 channel to 3 for MobileNet
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Download and load the training data
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"✅ Dataset loaded with {len(train_dataset)} training images.")

✅ Dataset loaded with 60000 training images.


In [3]:
import torch.optim as optim

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# Train for 1 epoch (just to test)
print("Starting training...")
model.train()
for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data.to(device), target.to(device)

    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()

    if batch_idx % 100 == 0:
        print(f"Batch {batch_idx}/{len(train_loader)} -> Loss: {loss.item():.4f}")

print("✅ Training completed for 1 epoch!")

Starting training...
Batch 0/1875 -> Loss: 2.3214
Batch 100/1875 -> Loss: 0.5912
Batch 200/1875 -> Loss: 0.5376
Batch 300/1875 -> Loss: 0.2431
Batch 400/1875 -> Loss: 0.3386
Batch 500/1875 -> Loss: 0.1932
Batch 600/1875 -> Loss: 0.6194
Batch 700/1875 -> Loss: 0.1922
Batch 800/1875 -> Loss: 0.3191
Batch 900/1875 -> Loss: 0.3029
Batch 1000/1875 -> Loss: 0.3641
Batch 1100/1875 -> Loss: 0.3721
Batch 1200/1875 -> Loss: 0.2393
Batch 1300/1875 -> Loss: 0.3152
Batch 1400/1875 -> Loss: 0.2308
Batch 1500/1875 -> Loss: 0.1434
Batch 1600/1875 -> Loss: 0.0730
Batch 1700/1875 -> Loss: 0.2010
Batch 1800/1875 -> Loss: 0.1650
✅ Training completed for 1 epoch!


In [4]:
torch.save(model.state_dict(), 'aster_classifier.pth')
print("✅ Model saved as aster_classifier.pth")

✅ Model saved as aster_classifier.pth
